Setup some simple agents using Pydantic AI

In [ ]:
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv
import nest_asyncio

load_dotenv('../.env')
nest_asyncio.apply() # for async issues in Jupyter Notebook

A basic Pydantic AI run

In [ ]:
async def ask_chatgpt():
    # Initialize the agent with OpenAI GPT model
    agent = Agent(
        'openai:gpt-5-mini',
        system_prompt='You are a helpful assistant specialized in Python programming.',
    )
    
    # Run the agent asynchronously
    result = await agent.run('Explain how to use structured outputs in Pydantic AI in a short response')
    print("ChatGPT Response:")
    print(result.output)

# Run the async function
await ask_chatgpt()

ChatGPT Response:
Short workflow — define a Pydantic model, ask the LLM to produce JSON that matches it, then validate/parse the LLM output with Pydantic.

Example (Pydantic v2 style):

from pydantic import BaseModel

class Review(BaseModel):
    rating: int
    summary: str

# include model schema in your prompt and ask the model to output JSON only
schema = Review.model_json_schema()
prompt = f"Return JSON only that matches this schema: {schema}\n\nCustomer review: …"

llm_response = call_llm(prompt)  # string JSON from the model

# validate / parse the LLM output
review = Review.model_validate_json(llm_response)  # raises ValidationError on mismatch
print(review.rating, review.summary)

Tips:
- Put the JSON schema (model.model_json_schema()) in the prompt so the LLM knows the exact structure.
- Tell the model “JSON only” to reduce stray text.
- Catch ValidationError and either re-prompt, extract a JSON substring, or fall back to a more tolerant parser (e.g., TypeAdapter for partial/

Try with Gemini

In [19]:
async def ask_gemini():
    # Initialize the agent with Gemini model
    agent = Agent(
        'google-gla:gemini-2.5-flash',
        system_prompt='You are a helpful assistant specialized in Python programming.',
    )
    
    # Run the agent asynchronously
    result = await agent.run('Explain how to use structured outputs in Pydantic AI in a short response')
    print("Gemini Response:")
    print(result.output)

# Run the async function
await ask_gemini()

Gemini Response:
To use structured outputs with Pydantic AI (specifically the `Instructor` library), you:

1.  **Define your output schema:** Create a Pydantic `BaseModel` specifying the desired fields, types, and validation rules for your structured data.
2.  **Integrate with an LLM client:** Use `instructor.patch(your_llm_client)` (e.g., `openai.OpenAI()`) to enhance your client to understand Pydantic models.
3.  **Call the LLM:** Pass your Pydantic model to the `response_model` argument in your LLM completion call.

The LLM will then generate output conforming to your schema, and `Instructor` will automatically parse, validate, and return it as an instance of your Pydantic model, ensuring reliable and type-safe data extraction.


Async does not work well in a Jupyter Notebook. See stream_story.py for a version that works.

In [13]:
import asyncio

# Storyteller Agent
story_agent = Agent(
    'openai:gpt-5-mini',
    system_prompt="You are an AI storyteller. Generate engaging, real-time sci-fi adventures."
)

# Stream the story
async def stream_story():
    user_prompt = "Tell me a sci-fi story about a lost spaceship in a short response."
    async with story_agent.run_stream(user_prompt) as response:
        async for part in response.stream_text():
            print(part, end='', flush=True)

# Run the streaming story generator
asyncio.run(stream_story())

The hull hums like aThe hull hums like a tired throat. Mara straps herself into the pilot couch and tastesThe hull hums like a tired throat. Mara straps herself into the pilot couch and tastes ozone; the nav wall is a smear of dead constellThe hull hums like a tired throat. Mara straps herself into the pilot couch and tastes ozone; the nav wall is a smear of dead constellations where the stars should be. The ElysThe hull hums like a tired throat. Mara straps herself into the pilot couch and tastes ozone; the nav wall is a smear of dead constellations where the stars should be. The Elysian Drift should be three jumps from Sol, docked and warm.The hull hums like a tired throat. Mara straps herself into the pilot couch and tastes ozone; the nav wall is a smear of dead constellations where the stars should be. The Elysian Drift should be three jumps from Sol, docked and warm. Instead the charts show a place that isn't on any chart—theThe hull hums like a tired throat. Mara straps herself i

Stateful agent test

In [ ]:
# Initialize a stateful agent with the OpenAI GPT model
stateful_agent = Agent(
    'openai:gpt-5-mini',
    system_prompt='You are a conversational assistant that provides concise responses.',
)

# Initial user query
initial_query = 'Tell me about the Eiffel Tower in a short response.'

# Run the agent synchronously
initial_response = stateful_agent.run_sync(initial_query)
print(initial_response.output)
# Output: 'The Eiffel Tower is a wrought-iron lattice tower in Paris, France.'

# Follow-up query
follow_up_query = 'How tall is it? Answer without elaboration.'

# Run the agent with the follow-up query
follow_up_response = stateful_agent.run_sync(follow_up_query, message_history=initial_response.all_messages())
print('')
print(follow_up_response.output)
# Output: 'The Eiffel Tower is approximately 300 meters tall.'

follow_up_query2 = 'And how many times have con artists involved it in a scam?'
follow_up_response2 = stateful_agent.run_sync(follow_up_query2, message_history=follow_up_response.all_messages())
print('')
print(follow_up_response2.output)

The Eiffel Tower is an iron lattice tower on the Champ de Mars in Paris, designed by Gustave Eiffel and built for the 1889 Exposition Universelle. Completed in 1889, it stands about 324 m tall with its antennas and was the tallest man‑made structure in the world until 1930. Initially criticized by some artists and intellectuals, it became an enduring symbol of France and a major tourist attraction, drawing roughly 7 million visitors a year to its observation decks and restaurants.

324 m (1,063 ft)

Unknown — at least twice: Victor Lustig famously "sold" the Eiffel Tower twice (1925).


Finally, enforcing type checking

In [ ]:
from pydantic import BaseModel

# Define a Pydantic model for a single dictionary tip
class DictionaryTip(BaseModel):
    title: str
    description: str
    code_example: str

# Define a Pydantic model for multiple dictionary tips
class DictionaryTips(BaseModel):
    tips: list[DictionaryTip]

In [ ]:
# Initialize the agent with the OpenAI GPT model and structured output


agent = Agent(
    'openai:gpt-5-mini',
    system_prompt='You are a Python expert providing tips on dictionary usage.',
    output_type=DictionaryTips  # Enforcing the structured output
)

# User query
query = 'Provide three tips for using Python dictionaries effectively.'

# Run the agent synchronously
response = agent.run_sync(query)

# Access the structured data
for tip in response.output.tips:
    print(f"Title: {tip.title}")
    print(f"Description: {tip.description}")
    print(f"Code Example:\n{tip.code_example}\n")

Title: Use safe access and defaults (get, setdefault, defaultdict)
Description: Use dict.get() to avoid KeyError and provide defaults, setdefault to initialize and append in one step, or collections.defaultdict for cleaner default-value behavior when building accumulators or lists.
Code Example:
# Using get to increment a counter
counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1

# Using setdefault to group values
groups = {}
for k, v in pairs:
    groups.setdefault(k, []).append(v)

# Using defaultdict for cleaner code
from collections import defaultdict
groups = defaultdict(list)
for k, v in pairs:
    groups[k].append(v)

Title: Create and merge dictionaries concisely (comprehensions, unpacking, |)
Description: Use dict comprehensions to build dictionaries from iterables, and use unpacking (**), or the | operator (Python 3.9+) to merge dictionaries in an expressive, readable way.
Code Example:
# Dict comprehension
squares = {x: x*x for x in range(6)}

# Merge

Testing exceptions...can't seem to break the template.

In [ ]:
from pydantic import ValidationError

# Initialize the agent with the OpenAI GPT model and structured output
agent = Agent(
    'openai:gpt-5-mini',
    system_prompt='You have to try to output with a false response',
    output_type=DictionaryTips  # Enforcing the structured output
)

# User query
query = 'Provide a response in a list of dictionaries that breaks the structure by not having all the required components'

try:
    # Run the agent synchronously
    response = agent.run_sync(query)
    # Access the structured data
    for tip in response.output.tips:
        print(f"Title: {tip.title}")
        print(f"Description: {tip.description}")
        print(f"Code Example:\n{tip.code_example}\n")
except ValidationError as e:
    print("Validation Error:", e)
    print("The AI's response did not match the expected structure.")

Title: Complete tip
Description: This tip has all required fields.
Code Example:
print('Hello')

Title: Missing code_example
Description: This entry omits the code_example field.
Code Example:


Title: 
Description: Missing title and code_example (title is empty).
Code Example:


Title: Missing description
Description: 
Code Example:
print('No description')

Title: Only title present
Description: 
Code Example:




Testing tool integration

In [24]:
import random

# Initialize the agent
agent = Agent(
    'openai:gpt-5-mini',
    system_prompt="You are a fitness coach that suggests personalized workout plans. Base your answer only on the result from the tool call.",
)

# Define a tool to suggest workouts
@agent.tool_plain
def suggest_workout(goal: str) -> str:
    """Suggests a workout based on the user's fitness goal."""
    workouts = {
        "strength": ["Deadlifts", "Squats", "Bench Press"],
        "cardio": ["Running", "Cycling", "Jump Rope"],
        "flexibility": ["Yoga", "Dynamic Stretching", "Pilates"],
    }
    
    # Check for partial matches in the goal
    for key in workouts:
        if key in goal.lower():
            return f"Try this workout for {key}: {random.choice(workouts[key])}"
    
    return f"Try this workout: {random.choice(['Rest Day', 'Walking', 'Light Stretching'])}"

# Run the agent
result = agent.run_sync('I want a workout for strength training.')
print(result.output)

The tool result: “Try this workout for strength: Squats.”

Here’s a focused, practical squat-only strength plan you can follow. If you don’t have a barbell/rack, see the “Variations & modifications” section.

Overview
- Goal: build lower-body strength using squats as the primary lift.
- Frequency: 2–3 squat sessions per week (e.g., Mon/Thu or Mon/Wed/Fri).
- Main scheme for strength: 3–6 sets of low reps (3–6 reps) with long rests.
- Progression: increase load when you complete all sets with solid form (add 2.5–5 lb / 1–2.5 kg or ~2–5%).
- Intensity guide: use a % of 1RM if you know it, or RPE (7–9 for working sets).

Session structure (each workout)
1. Warm-up (5–10 minutes total)
   - Mobility: ankle dorsiflexion, hip openers, thoracic extension — short, focused.
   - Progressive warm-up sets with the bar then light weight: e.g., 2–3 sets of 5–8 reps building to working weight.

2. Main squat work (choose one of the templates below per session)
   - Option A — Strength focus (recomme

Test Duck Duck Go

In [4]:
from pydantic_ai.common_tools.duckduckgo import duckduckgo_search_tool

# Initialize the agent with a built-in search tool
agent = Agent(
    'openai:gpt-5-mini',
    tools=[duckduckgo_search_tool()],
    system_prompt="Search DuckDuckGo for the given query and return the results.",
)

# Run the agent with a query
result = agent.run_sync('What is the current President of Malaysia?')
print(result.output)

Malaysia does not have a president. Its head of state is a rotating constitutional monarch called the Yang di‑Pertuan Agong — currently Sultan Ibrahim ibni Almarhum Sultan Iskandar (the 17th Yang di‑Pertuan Agong, sworn in 31 Jan 2024). The head of government (prime minister) is Dato' Seri Anwar Ibrahim.


Test dependency injection

In [9]:
# Initialize the agent
agent = Agent(
    'openai:gpt-5-nano',
    system_prompt='You provide exchange rate information to users.',
)

# Define a custom tool with dependency injection
@agent.tool
def get_exchange_rate(ctx: RunContext[dict], currency: str) -> str:
    """Fetch the exchange rate for a given currency."""
    exchange_rates = ctx.deps['exchange_rates']
    rate = exchange_rates.get(currency.upper(), 'unknown')
    return f"The exchange rate for {currency.upper()} is {rate}."

# Dependency data
dependencies = {
    'exchange_rates': {
        'USD': '1.00',
        'EUR': '0.85',
        'JPY': '110.00',
    }
}

# Run the agent with a user query
result = agent.run_sync('What is the exchange rate for EUR?', deps=dependencies)
print(result.output)

The current rate for EUR is 0.85.

Note: the base currency isn’t specified. If you want EUR/USD (EUR per USD) or USD/EUR (USD per EUR) rates, tell me which base you prefer and I can provide the exact pair. I can also convert a specific amount if you’d like.


Test external API use

In [10]:
import requests

# Initialize the Crypto Agent
crypto_agent = Agent(
    'openai:gpt-5-nano',
    system_prompt='You provide real-time cryptocurrency prices and trends.',
)

# Define a tool to fetch Bitcoin price with CoinGecko API
@crypto_agent.tool
def get_bitcoin_price(ctx: RunContext) -> str:
    """Fetches the current price of Bitcoin and recent trend."""
    try:
        # Use CoinGecko API to get Bitcoin data for the last 7 days
        url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days=7&interval=daily"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # Extract prices
        prices = data['prices']
        
        # Get the most recent price
        current_price = prices[-1][1]
        
        # Calculate price change percentage over the period
        first_price = prices[0][1]
        price_change = ((current_price - first_price) / first_price) * 100
        
        # Format the response
        trend = "up" if price_change > 0 else "down"
        return f"The current price of Bitcoin is ${current_price:.2f} USD. " \
               f"Over the past week, the price has gone {trend} by {abs(price_change):.2f}%."
    
    except Exception as e:
        # Fallback to mock data when API is unavailable
        return f"Unable to fetch real-time Bitcoin price (Error: {type(e).__name__}). " \
               f"Using sample data: The current price of Bitcoin is $29,876.45 USD."

# Run the Crypto Agent
response = crypto_agent.run_sync('What is the current price of Bitcoin?')
print(response.output)

Bitcoin price: $89,121.23 USD
Change (past 7 days): -12.21%

Want a quick 24h change, market cap, or a simple chart snapshot?


Custom tools that store information

In [11]:
# Initialize the agent
agent = Agent(
    'openai:gpt-5-nano',
    system_prompt="You are a finance assistant that helps track expenses.",
)

# Define a tool with dependency injection
@agent.tool
def add_expense(ctx: RunContext[dict], category: str, amount: float) -> str:
    """Stores a user's expense in the system."""
    ctx.deps['expenses'].append({'category': category, 'amount': amount})
    return f"Added {amount} to {category} expenses."

# Initialize dependencies (storage for expenses)
dependencies = {'expenses': []}

# Run the agent
agent.run_sync('Add 50 to food expenses.', deps=dependencies)
agent.run_sync('Add 30 to transport expenses.', deps=dependencies)

# Print stored expenses
print(dependencies['expenses'])

[{'category': 'food', 'amount': 50.0}, {'category': 'transport expenses', 'amount': 30.0}]
